# Alternatives to the RBF SVM

The shipped letter model is an RBF SVM on 74 geometry features. This notebook asks whether a
different classifier does better **on the same leave-one-signer-out folds**, and whether a
better letter model actually moves word accuracy, which is what the demo shows.

Three things are reported for each model, because they disagree:

* **accuracy**: share of holds right. Dominated by the common letters.
* **macro recall**: mean of per-letter recall over the 40 letters. Treats rare letters equally.
* **log-loss**: quality of the probabilities. This is what the word decoder consumes.

Candidates and their settings are defined in `models.py` and were fixed before looking at
held-out results. Section 2 then selects among them **inside the training signers**, because
picking the best of several models on five test folds is optimistic.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd
import evaluation as ev, end_to_end as e2e, word_decode as wd, models

pd.set_option("display.width", 220)
P, M = ev.load_dataset()
F = ev.geometry_features(P)
y, sg = M.letter.to_numpy(), M.signer.to_numpy()
CL = sorted(set(y))
SEEDS = range(3)

def loso_proba(make, seed, F=F, y=y, sg=sg):
    """LOSO predictions and probabilities (columns aligned to CL)."""
    pred = np.empty(len(y), dtype=object); Pr = np.zeros((len(y), len(CL)))
    for s in sorted(set(sg)):
        tr, te = sg != s, sg == s
        m = make(seed).fit(F[tr], y[tr])
        pred[te] = m.predict(F[te])
        pr = m.predict_proba(F[te])
        for j, c in enumerate(m.classes_): Pr[te, CL.index(c)] = pr[:, j]
    return pred, Pr

def metrics(pred, Pr, y=y, sg=sg):
    ok = (pred == y).astype(float)
    p_true = Pr[np.arange(len(y)), [CL.index(t) for t in y]]
    return {"accuracy %": 100 * ev.per_fold(ok, sg).mean(),
            "macro recall %": 100 * pd.Series(ok).groupby(y).mean().mean(),
            "log-loss": float(-np.log(np.maximum(p_true, 1e-6)).mean())}

## 1. All candidates, fixed settings, three seeds

In [2]:
rows, keep = [], {}
for name, make in models.CANDIDATES.items():
    t = time.time(); ms = []
    for seed in SEEDS:
        pred, Pr = loso_proba(make, seed); ms.append(metrics(pred, Pr))
        if seed == 0: keep[name] = pred
    D = pd.DataFrame(ms)
    rows.append({"model": name, **D.mean().round(3).to_dict(),
                 "accuracy range over seeds": round(D["accuracy %"].max() - D["accuracy %"].min(), 1),
                 "seconds": round(time.time() - t)})
T1 = pd.DataFrame(rows).set_index("model")
print(T1.round(2).to_string())

                                  accuracy %  macro recall %  log-loss  accuracy range over seeds  seconds
model                                                                                                     
SVM-RBF (shipped)                      26.52           23.71      2.76                        0.0       34
ExtraTrees                             31.73           26.72      2.87                        0.1       32
MLP 128, alpha=3                       33.50           17.36      2.62                        0.4      107
LDA, shrinkage                         30.65           22.57      3.90                        0.0        1
kNN, k=15                              31.06           17.68      5.57                        0.0        7
ensemble: SVM+MLP+ExtraTrees+LDA       33.21           23.01      2.60                        0.4      175


In [3]:
maj = ev.majority_correct(y, sg)
base = (keep["SVM-RBF (shipped)"] == y).astype(float)
print(pd.DataFrame([ev.summarise(n, (p == y).astype(float), sg, baseline=base) for n, p in keep.items()])
        .drop(columns=["n"]).round(3).to_string(index=False))
print("\n'vs baseline' is the gain over the shipped SVM, paired over the five signers, seed 0.")

                          metric  pooled %  mean of folds %  t half-width  boot lo %  boot hi %  vs baseline (pts)  paired t p
               SVM-RBF (shipped)    26.394           26.519         5.682     22.555     30.379              0.000         NaN
                      ExtraTrees    31.822           31.771         6.037     27.956     35.833              5.251       0.001
                MLP 128, alpha=3    33.606           33.652         3.687     31.121     36.136              7.132       0.001
                  LDA, shrinkage    30.260           30.648         6.299     26.012     34.119              4.128       0.067
                       kNN, k=15    31.227           31.057         4.547     28.248     34.513              4.538       0.016
ensemble: SVM+MLP+ExtraTrees+LDA    33.309           33.363         3.187     31.520     35.614              6.844       0.007

'vs baseline' is the gain over the shipped SVM, paired over the five signers, seed 0.


## 2. Honest selection: choose the model inside the training signers

For each held-out signer, every candidate is scored by leave-one-signer-out over the four
training signers, the best is chosen, refitted on all four, and tested once.

In [4]:
def nested_select(criterion):
    pred = np.empty(len(y), dtype=object); chosen = {}
    for s in sorted(set(sg)):
        tr, te = sg != s, sg == s
        best = None
        for name, make in models.CANDIDATES.items():
            p, Pr = loso_proba(make, 0, F[tr], y[tr], sg[tr])
            m = metrics(p, Pr, y[tr], sg[tr])
            val = m[criterion] if criterion != "log-loss" else -m[criterion]
            if best is None or val > best[0]: best = (val, name)
        chosen[s] = best[1]
        pred[te] = models.CANDIDATES[best[1]](0).fit(F[tr], y[tr]).predict(F[te])
    return pred, chosen

sel = {}
for crit in ("accuracy %", "macro recall %"):
    pred, chosen = nested_select(crit)
    sel[crit] = pred
    ok = (pred == y).astype(float)
    print(f"selected by inner {crit}:")
    print("   chosen per held-out signer:", chosen)
    r = ev.summarise("nested-selected", ok, sg, baseline=base)
    print(f"   accuracy {r['mean of folds %']:.1f}% (CI {r['boot lo %']:.1f} to {r['boot hi %']:.1f}), "
          f"macro recall {100*pd.Series(ok).groupby(y).mean().mean():.1f}%, "
          f"gain over shipped SVM {r['vs baseline (pts)']:+.1f} pts (paired p = {r['paired t p']:.3f})\n")

selected by inner accuracy %:
   chosen per held-out signer: {'signer1-F': 'MLP 128, alpha=3', 'signer2-F': 'MLP 128, alpha=3', 'signer3-F': 'ensemble: SVM+MLP+ExtraTrees+LDA', 'signer4-L': 'ensemble: SVM+MLP+ExtraTrees+LDA', 'signer5-R': 'MLP 128, alpha=3'}
   accuracy 32.6% (CI 30.4 to 35.1), macro recall 18.3%, gain over shipped SVM +6.1 pts (paired p = 0.005)



selected by inner macro recall %:
   chosen per held-out signer: {'signer1-F': 'ExtraTrees', 'signer2-F': 'ExtraTrees', 'signer3-F': 'ExtraTrees', 'signer4-L': 'ExtraTrees', 'signer5-R': 'ExtraTrees'}
   accuracy 31.8% (CI 28.0 to 35.8), macro recall 27.3%, gain over shipped SVM +5.3 pts (paired p = 0.001)



## 3. Which letters change?

Per-letter recall of the shipped SVM against ExtraTrees and the MLP, for the letters the audit
flagged and the six that the SVM never recognises.

In [5]:
focus = ["م", "ح", "ر", "س", "ص", "ڑ", "خ", "پ", "ش", "چ"]
R = pd.DataFrame({n: pd.Series((p == y).astype(float)).groupby(y).mean() for n, p in keep.items()})
R.insert(0, "n", pd.Series(y).value_counts())
print((100 * R.loc[focus].drop(columns="n")).round(0).astype(int).assign(n=R.loc[focus, "n"]).to_string())
print("\nletters with recall 0:", {n: int((R[n] == 0).sum()) for n in keep})

   SVM-RBF (shipped)  ExtraTrees  MLP 128, alpha=3  LDA, shrinkage  kNN, k=15  ensemble: SVM+MLP+ExtraTrees+LDA    n
م                 16          18                14               7         11                                 9   57
ح                  8          19                15              27         12                                23   26
ر                 22          33                47              41         41                                43  120
س                 37          56                25              15         27                                27   52
ص                  0           0                 0              25          0                                25    4
ڑ                  0           0                 0               0          0                                 0    4
خ                  0          40                 0              20          0                                20    5
پ                  0           0                 0              

## 4. Does a better letter model move word accuracy?

End to end on the free holds, with the alignment decoder and likelihood ratio from
`alignment_decoder.ipynb`. Decoder costs are tuned on the training signers for each model
separately. Seed 0.

In [6]:
def words(make):
    nested, lex = e2e.nested_clips(seed=0, make_model=make)
    parts = []
    for s, d in nested.items():
        ins, dele = wd.tune_aligned(d["tune"], lex, "free", True)
        dec = lambda probs, classes, lex, clip, a=ins, b=dele: wd.decode_aligned(probs, classes, lex, clip["prior"], a, b)
        parts.append(e2e.word_table(d["test"], lex, dec, "free"))
    return pd.concat(parts, ignore_index=True)

W, rows = {}, []
for name in ["SVM-RBF (shipped)", "ExtraTrees", "MLP 128, alpha=3", "ensemble: SVM+MLP+ExtraTrees+LDA"]:
    t = time.time(); W[name] = words(models.CANDIDATES[name])
    sgw = W[name].signer.to_numpy()
    r1 = ev.summarise("", W[name].top1.to_numpy(), sgw); r3 = ev.summarise("", W[name].top3.to_numpy(), sgw)
    rows.append({"letter model": name, "word top-1 %": r1["mean of folds %"], "CI lo": r1["boot lo %"], "CI hi": r1["boot hi %"],
                 "word top-3 %": r3["mean of folds %"], "top-3 lo": r3["boot lo %"], "top-3 hi": r3["boot hi %"],
                 "seconds": round(time.time() - t)})
print(pd.DataFrame(rows).round(1).to_string(index=False))

                    letter model  word top-1 %  CI lo  CI hi  word top-3 %  top-3 lo  top-3 hi  seconds
               SVM-RBF (shipped)          74.8   55.9   85.9          88.2      75.2      95.3      122
                      ExtraTrees          71.1   54.8   82.9          85.9      75.1      92.5      228
                MLP 128, alpha=3          79.2   61.4   91.0          90.0      79.3      96.5      157
ensemble: SVM+MLP+ExtraTrees+LDA          81.4   65.3   91.0          91.7      80.2      98.0      384


In [7]:
print("word top-1 % per held-out signer:")
print(pd.DataFrame({n: 100 * t.groupby("signer").top1.mean() for n, t in W.items()}).round(1).to_string())
b = W["SVM-RBF (shipped)"]
for n, t in W.items():
    if n == "SVM-RBF (shipped)": continue
    r = ev.summarise(n, t.top1.to_numpy(), t.signer.to_numpy(), baseline=b.top1.to_numpy())
    print(f"{n}: {r['vs baseline (pts)']:+.1f} pts top-1 over the SVM, paired p = {r['paired t p']:.3f}")

word top-1 % per held-out signer:
           SVM-RBF (shipped)  ExtraTrees  MLP 128, alpha=3  ensemble: SVM+MLP+ExtraTrees+LDA
signer                                                                                      
signer1-F               85.7        79.6              87.8                              91.8
signer2-F               83.7        85.7              89.8                              87.8
signer3-F               75.6        66.7              73.3                              82.2
signer4-L               40.8        42.9              49.0                              53.1
signer5-R               88.5        80.8              96.2                              92.3
ExtraTrees: -3.7 pts top-1 over the SVM, paired p = 0.195
MLP 128, alpha=3: +4.4 pts top-1 over the SVM, paired p = 0.091
ensemble: SVM+MLP+ExtraTrees+LDA: +6.6 pts top-1 over the SVM, paired p = 0.012


## Reading

* Read sections 1 and 2 together. Section 1 shows which models look better with settings fixed
  in advance; section 2 is the number to quote, because the choice is made without the test
  signer.
* Accuracy and macro recall rank the models differently. A model can gain accuracy by doing
  better on the common letters while losing the rare ones; section 3 shows which.
* Section 4 is the test that matters for the demo: letter accuracy is an intermediate
  quantity, word accuracy is the product.
* All of this is five signers. Differences of a few points between models are inside the
  interval and should not be reported as improvements.